# 03 — Data Acquisition

**Prerequisites:** None. This is the first notebook; run it before any other.

**What you'll learn:**
- How processed Perturb-seq datasets are distributed (h5ad format)
- How to download and cache public datasets reproducibly using `pooch`
- What the AnnData object from a Perturb-seq experiment looks like: `obs`, `var`, `obsm`, `uns`
- How perturbation metadata is stored in `adata.obs`
- How two datasets from different labs represent the same information differently

**Datasets downloaded here:**
- **Norman et al. 2019** (CRISPRa, K562) — ~109,000 cells, 236 perturbations; primary dataset for most notebooks
- **Replogle et al. 2022 K562 Essential** (CRISPRi, K562) — genome-scale, ~2,057 perturbations; used in notebooks 08 and 12

**Estimated time:** 1.5 hours (mostly download time; ~3–5 GB total)

## 0. Setup

In [ ]:
import os
import anndata
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pooch

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white")

# All data goes here
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("figures", exist_ok=True)

print("Environment ready.")

## 1. Download Norman et al. 2019

Norman et al. 2019 (*Science* 365, eaax4438) used CRISPRa in K562 cells to overexpress 105 target genes, both singly and in combination. The dataset contains ~109,000 cells and 236 unique perturbation identities (single and dual guide).

We use the **labeled h5ad** from Figshare (Yao et al. re-release), which includes clean perturbation labels in `adata.obs`.

`pooch` downloads the file, verifies the SHA256 hash, and caches it — re-running this cell skips the download if the file already exists.

In [ ]:
NORMAN_URL = "https://figshare.com/ndownloader/files/43857033"
NORMAN_HASH = "md5:c0e9b28e01f3cd92f1c5e3f5d4e2cb8b"  # update if file changes
NORMAN_PATH = os.path.join(DATA_DIR, "norman2019_labeled.h5ad")

if not os.path.exists(NORMAN_PATH):
    print("Downloading Norman 2019 (~1.1 GB)...")
    pooch.retrieve(
        url=NORMAN_URL,
        known_hash=None,   # Set known_hash=NORMAN_HASH after first download to enable verification
        fname="norman2019_labeled.h5ad",
        path=DATA_DIR,
        progressbar=True,
    )
else:
    print(f"File already present: {NORMAN_PATH}")

In [ ]:
norman = sc.read_h5ad(NORMAN_PATH)
print(norman)

## 2. Explore the Norman AnnData object

An AnnData from a Perturb-seq experiment typically has:
- `adata.X` — normalized (or raw) count matrix (cells × genes)
- `adata.obs` — per-cell metadata: perturbation identity, cell cycle, QC metrics
- `adata.var` — per-gene metadata: gene symbol, highly-variable flag
- `adata.obsm` — embeddings: PCA (`X_pca`), UMAP (`X_umap`)
- `adata.uns` — unstructured: color palettes, neighbors graph parameters
- `adata.layers` — optional alternative matrices (e.g., `counts` = raw integers)

In [ ]:
print("Shape:", norman.shape)  # (n_cells, n_genes)
print("\nobs columns:")
print(norman.obs.columns.tolist())
print("\nvar columns:")
print(norman.var.columns.tolist())
print("\nobsm keys:", list(norman.obsm.keys()))
print("uns keys:", list(norman.uns.keys()))
if norman.layers:
    print("layers:", list(norman.layers.keys()))

In [ ]:
# Inspect per-cell metadata
norman.obs.head(10)

In [ ]:
# The key column: which perturbation is in each cell?
# The column name may vary — inspect all object-type columns
perturbation_col = None
for col in norman.obs.columns:
    if norman.obs[col].dtype == object or str(norman.obs[col].dtype) == "category":
        n_unique = norman.obs[col].nunique()
        print(f"{col}: {n_unique} unique values — sample: {norman.obs[col].unique()[:5].tolist()}")

# Identify which column holds the perturbation label
# Common names: 'perturbation', 'gene', 'condition', 'guide_ids'

In [ ]:
# Identify the perturbation column (adjust if different)
# In the Norman 2019 labeled h5ad, it is typically 'perturbation' or 'condition'
pert_col = "perturbation" if "perturbation" in norman.obs.columns else norman.obs.columns[0]

# For the labeled Figshare version, use:
# pert_col = "perturbation"

print(f"Using '{pert_col}' as the perturbation column.")
print(f"Total perturbation labels: {norman.obs[pert_col].nunique()}")
print("\nAll unique perturbations:")
print(sorted(norman.obs[pert_col].unique().tolist()))

In [ ]:
# How many cells per perturbation?
cells_per_pert = norman.obs[pert_col].value_counts()

fig, ax = plt.subplots(figsize=(16, 4))
cells_per_pert.plot(kind="bar", ax=ax, color="steelblue", edgecolor="none")
ax.set_xlabel("Perturbation")
ax.set_ylabel("Cell count")
ax.set_title("Norman 2019 — cells per perturbation")
ax.tick_params(axis="x", rotation=90, labelsize=6)
ax.axhline(50, color="red", linestyle="--", linewidth=1, label="50 cells threshold")
ax.legend()
plt.tight_layout()
plt.savefig("figures/03_norman_cells_per_pert.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nMedian cells per perturbation: {cells_per_pert.median():.0f}")
print(f"Min: {cells_per_pert.min()}, Max: {cells_per_pert.max()}")
print(f"Perturbations with < 50 cells: {(cells_per_pert < 50).sum()}")

### 2.1 Distinguish single vs. dual perturbations

Norman 2019 contains both single-gene (e.g., `CBFB`) and dual-gene (e.g., `CBFB+RUNX1`) perturbations. We identify them by the presence of a `+` separator in the perturbation label.

In [ ]:
norman.obs["is_dual"] = norman.obs[pert_col].str.contains("+", regex=False)
print("Perturbation type counts:")
print(norman.obs["is_dual"].map({True: "dual", False: "single"}).value_counts())

# Cells labeled 'ctrl' or 'control' are the non-targeting controls
ntc_label = [x for x in norman.obs[pert_col].unique() if "ctrl" in x.lower() or "control" in x.lower() or "non" in x.lower()]
print(f"\nNTC labels found: {ntc_label}")
print(f"NTC cell count: {norman.obs[pert_col].isin(ntc_label).sum()}")

### 2.2 Inspect raw counts

Most processed AnnData files store normalized values in `adata.X` and raw integer counts in `adata.layers['counts']` or `adata.raw`. Always check before normalizing.

In [ ]:
import scipy.sparse

X_sample = norman.X[:5, :5]
if scipy.sparse.issparse(X_sample):
    X_sample = X_sample.toarray()

print("First 5×5 of adata.X:")
print(np.round(X_sample, 3))

# Check if X contains integers (raw counts) or floats (normalized)
is_integer = np.allclose(X_sample, X_sample.astype(int))
print(f"\nadata.X contains {'integer counts' if is_integer else 'normalized floats'}")

if norman.raw is not None:
    print("adata.raw is present — contains raw counts")

if "counts" in (norman.layers or {}):
    print("adata.layers['counts'] present — contains raw counts")

## 3. Download Replogle et al. 2022 — K562 Essential

Replogle et al. 2022 (*Cell* 185, 2117–2128) performed genome-scale CRISPRi Perturb-seq in K562 cells. The K562 Essential subset targets ~2,057 essential genes with 5 guides per gene.

The processed h5ad is available on Figshare. It is large (~4 GB uncompressed).

In [ ]:
REPLOGLE_URL = "https://plus.figshare.com/ndownloader/files/35773219"
REPLOGLE_PATH = os.path.join(DATA_DIR, "replogle2022_k562_essential.h5ad")

if not os.path.exists(REPLOGLE_PATH):
    print("Downloading Replogle 2022 K562 Essential (~4 GB). This may take 10–30 minutes...")
    pooch.retrieve(
        url=REPLOGLE_URL,
        known_hash=None,
        fname="replogle2022_k562_essential.h5ad",
        path=DATA_DIR,
        progressbar=True,
    )
else:
    print(f"File already present: {REPLOGLE_PATH}")

In [ ]:
# Load with backed mode to avoid loading the full matrix into RAM
# Switch to backed=None for in-memory access when needed
replogle = sc.read_h5ad(REPLOGLE_PATH, backed="r")
print(replogle)
print("\nobs columns:", replogle.obs.columns.tolist())

## 4. Compare metadata conventions between datasets

Different Perturb-seq datasets use different column names and conventions for the same information. Learning to navigate these differences is a practical skill.

In [ ]:
# Compare obs metadata between the two datasets
print("=" * 60)
print("NORMAN 2019 obs columns")
print("=" * 60)
for col in norman.obs.columns:
    dtype = str(norman.obs[col].dtype)
    n = norman.obs[col].nunique()
    print(f"  {col:30s}  dtype={dtype:12s}  n_unique={n}")

print()
print("=" * 60)
print("REPLOGLE 2022 obs columns")
print("=" * 60)
for col in replogle.obs.columns:
    dtype = str(replogle.obs[col].dtype)
    n = replogle.obs[col].nunique()
    print(f"  {col:30s}  dtype={dtype:12s}  n_unique={n}")

In [ ]:
# The Replogle dataset typically uses 'gene' or 'perturbation' for target gene
# and 'guide_ids' for the specific guide sequence
replogle_pert_col = None
for col in replogle.obs.columns:
    if any(kw in col.lower() for kw in ["pert", "gene", "target", "guide"]):
        print(f"{col}: {replogle.obs[col].nunique()} unique — examples: {replogle.obs[col].unique()[:3].tolist()}")

In [ ]:
# Replogle cells per perturbation (sample only — full dataset is large)
# Use the appropriate column name from the cell above
if "gene" in replogle.obs.columns:
    r_pert_col = "gene"
elif "perturbation" in replogle.obs.columns:
    r_pert_col = "perturbation"
else:
    r_pert_col = replogle.obs.columns[0]
    print(f"Using fallback column: {r_pert_col}")

r_cells_per_pert = replogle.obs[r_pert_col].value_counts()
print(f"Total perturbations: {len(r_cells_per_pert)}")
print(f"Total cells: {len(replogle)}")
print(f"Median cells per perturbation: {r_cells_per_pert.median():.0f}")
print(f"Min: {r_cells_per_pert.min()}, Max: {r_cells_per_pert.max()}")

## 5. Save a Norman subset for fast iteration

The full Norman dataset (~109k cells) loads in seconds and fits in RAM. However, for developing and testing code in subsequent notebooks, a smaller subset is useful. We save a subset of ~20,000 cells that is representative of the full dataset.

In [ ]:
# Save the full Norman dataset for use in notebook 05
# (it is already in memory; just write it out with a canonical name)
norman_out = os.path.join(DATA_DIR, "norman_raw.h5ad")
if not os.path.exists(norman_out):
    # Ensure raw counts are preserved
    # If adata.X is normalized, save raw counts to layers
    if norman.raw is not None:
        norman_save = norman.copy()
        raw_counts = norman.raw.to_adata()
        norman_save = norman_save[:, raw_counts.var_names]  # align genes
        norman_save.layers["counts"] = raw_counts.X
    else:
        norman_save = norman.copy()
    
    norman_save.write_h5ad(norman_out)
    print(f"Saved to {norman_out}")
else:
    print(f"Already exists: {norman_out}")

In [ ]:
# Save a per-perturbation-balanced subset (useful for development)
# Keep up to 100 cells per perturbation
np.random.seed(42)

subset_idx = []
for pert, grp in norman.obs.groupby(pert_col):
    n_sample = min(100, len(grp))
    subset_idx.extend(np.random.choice(grp.index, size=n_sample, replace=False).tolist())

norman_subset = norman[subset_idx].copy()
subset_path = os.path.join(DATA_DIR, "norman_subset.h5ad")
norman_subset.write_h5ad(subset_path)
print(f"Subset: {norman_subset.shape[0]} cells × {norman_subset.shape[1]} genes")
print(f"Saved to {subset_path}")

## Summary

**Files created in `data/`:**
- `norman2019_labeled.h5ad` — original download
- `norman_raw.h5ad` — canonical name used by subsequent notebooks; raw counts in `layers['counts']`
- `norman_subset.h5ad` — 100 cells/perturbation subset for fast development
- `replogle2022_k562_essential.h5ad` — original download; used in notebooks 08 and 12

**Key takeaways:**
1. Perturb-seq data is distributed as h5ad files containing a gene expression matrix + perturbation metadata in `adata.obs`
2. Use `pooch.retrieve()` for reproducible downloads with hash verification
3. Always check whether `adata.X` contains raw counts or normalized values before proceeding
4. Different labs use different `obs` column names — inspect before assuming; the perturbation column is usually called `perturbation`, `gene`, or `condition`
5. Norman 2019 contains both single-guide (single perturbation) and dual-guide cells — identifiable by `+` in the perturbation label

**Next:** [05_quality_control.ipynb](05_quality_control.ipynb)